In [18]:

using JSON
using JuMP
using GLPK
using CSV
using DataFrames
import MathOptInterface as MOI
using HiGHS
#using SCIP
#using Cbc



# CSV

In [19]:
# CSV ---------------------------------------------------------------------------------------------------
function criar_arquivo_csv(nome_arquivo::AbstractString)
  if isfile(nome_arquivo)
    rm(nome_arquivo)
  end
  df_vazio = DataFrame(
    nomearq = String[],
    modelo = String[],
    custosub = String[],
    custocustosub = String[],
    custototal = Float64[],
    tempo = Float64[]
  )
  CSV.write(nome_arquivo, df_vazio)
  return nome_arquivo
end

function salvar_resultados_csv(
  nome_arquivo::AbstractString;
  nomearq::AbstractString,
  modelo::AbstractString,
  custosub,
  custocustosub,
  custototal::Real,
  tempo::Real
)

  linha = DataFrame(
    nomearq = [nomearq],
    modelo = [modelo],
    custosub = [JSON.json(custosub)],
    custocustosub = [JSON.json(custocustosub)],
    custototal = [Float64(custototal)],
    tempo = [Float64(tempo)]
  )

  if isfile(nome_arquivo)
      CSV.write(nome_arquivo, linha; append=true)
  else
      CSV.write(nome_arquivo, linha)
  end
end

function carregar_dados(json_path)
  dados = JSON.parsefile(json_path)
  return dados
end

function ler_chave(dados, chave)
    if !haskey(dados, chave)
        error("'$chave' não encontrada no JSON.")
    end
    return dados[chave]
end

function extrair_rotas(x)
    n = size(x, 1)
    if ndims(x) == 2
        rotas = []
        for j in 2:n
            if x[1, j] > 0.5
                rota = [1, j]
                atual = j
                while atual != 1
                    for k in 1:n
                        if x[atual, k] > 0.5
                            push!(rota, k)
                            atual = k
                            break
                        end
                    end
                end
                push!(rotas, rota)
            end
        end
        return rotas
    elseif ndims(x) == 3
        K = size(x, 3)
        rotas = []
        for k in 1:K
            for j in 2:n
                if x[1, j, k] > 0.5
                    rota = [1, j]
                    atual = j
                    while atual != 1
                        for i in 1:n
                            if x[atual, i, k] > 0.5
                                push!(rota, i)
                                atual = i
                                break
                            end
                        end
                    end
                    push!(rotas, rota)
                    break
                end
            end
        end
        return rotas
    end
end

function calc_custo(rota, matriz_custo)
    custo_total = 0.0
    for i in 1:length(rota)-1
        custo_total += matriz_custo[rota[i], rota[i+1]]
    end
    return custo_total
end

calc_custo (generic function with 1 method)

# VRP CLÁSSICO (HIGHS)



In [20]:
# HiGHS ---------------------------------------------------------------------------------------------------

function pipeline_highs(file_path, modelo, nome_modelo, tw=false)
    dados = carregar_dados(file_path)
    matriz_custo = ler_chave(dados, "matriz_custos")
    matriz_custo = Float64.(hcat(matriz_custo...)')
    matriz_tempo = ler_chave(dados, "matriz_tempos")
    matriz_tempo = Float64.(hcat(matriz_tempo...)')

    custo_geral = 0.0
    tempo_inicio = time()
    if tw
        x = modelo(matriz_custo, matriz_tempo)
    else
        x = modelo(matriz_custo)
    end
    tempo_fim = time()
    tempo = tempo_fim - tempo_inicio
    rota_local = extrair_rotas(x)
    println("Rotas HiGHS: ", rota_local)
    rota_json = [[r-1 for r in rota] for rota in rota_local]
    salvar_resultados_csv(
        arquivo_csv;
        nomearq = file_path,
        modelo = nome_modelo,
        custosub = rota_json,
        custocustosub = [calc_custo(rota, matriz_custo) for rota in rota_local],
        custototal = sum(calc_custo(rota, matriz_custo) for rota in rota_local),
        tempo = tempo
    )
end

function highs_tsp_classico(matriz_custo; k=5)
    n = size(matriz_custo, 1)

    model = Model(HiGHS.Optimizer)
    set_silent(model)

    @variable(model, x[1:n, 1:n], Bin)
    @variable(model, 0 <= u[1:n] <= n)

    @constraint(model, [i in 1:n], x[i,i] == 0)

    @objective(model, Min,
        sum(matriz_custo[i,j] * x[i,j] for i in 1:n, j in 1:n)
    )

    @constraint(model, [i in 2:n],
        sum(x[i,j] for j in 1:n if j != i) == 1
    )

    @constraint(model, [j in 2:n],
        sum(x[i,j] for i in 1:n if i != j) == 1
    )

    for i in 2:n, j in 2:n
        if i != j
            @constraint(model, u[i] - u[j] + n*x[i,j] <= n-1)
        end
    end

    optimize!(model)
    return value.(x)
end

function highs_tsp_mtz_1(matriz_custo; t=1)
	n = size(matriz_custo, 1)
	p = n - 1
	model = Model(HiGHS.Optimizer)
	set_silent(model)
	@variable(model, x[1:n, 1:n], Bin)
	@variable(model, 1 <= u[2:n] <= p)
	@constraint(model, [i in 1:n], x[i,i] == 0)
	@objective(model, Min, sum(matriz_custo[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))
	@constraint(model, [i in 2:n], sum(x[i,j] for j in 1:n if j != i) == 1)
	@constraint(model, [j in 2:n], sum(x[i,j] for i in 1:n if i != j) == 1)
	@constraint(model, sum(x[1,j] for j in 2:n) == t)
	@constraint(model, sum(x[i,1] for i in 2:n) == t)
	@constraint(model, [i in 2:n, j in 2:n; i != j], u[i] - u[j] + p * x[i,j] <= p - 1)
	optimize!(model)
	return value.(x)
end


function highs_tsp_mtz_5(matriz_custo; t=5)
	n = size(matriz_custo, 1)
	p = n - 1
	model = Model(HiGHS.Optimizer)
	set_silent(model)
	@variable(model, x[1:n, 1:n], Bin)
	@variable(model, 1 <= u[2:n] <= p)
	@constraint(model, [i in 1:n], x[i,i] == 0)
	@objective(model, Min, sum(matriz_custo[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))
	@constraint(model, [i in 2:n], sum(x[i,j] for j in 1:n if j != i) == 1)
	@constraint(model, [j in 2:n], sum(x[i,j] for i in 1:n if i != j) == 1)
	@constraint(model, sum(x[1,j] for j in 2:n) == t)
	@constraint(model, sum(x[i,1] for i in 2:n) == t)
	@constraint(model, [i in 2:n, j in 2:n; i != j], u[i] - u[j] + p * x[i,j] <= p - 1)
	optimize!(model)
	return value.(x)
end

function highs_vrp_mtz_TW(matriz_custo, matriz_tempo; jornada=480, coleta=60)
    n = size(matriz_custo, 1)
    p = n - 1
    K = 1:p
    model = Model(HiGHS.Optimizer)
    set_silent(model)
    @variable(model, x[1:n, 1:n, K], Bin)
    @variable(model, 1 <= u[2:n, K] <= p)
    @constraint(model, [i in 1:n, k in K], x[i,i,k] == 0)
    @objective(model, Min, sum(matriz_custo[i,j] * x[i,j,k] for i in 1:n, j in 1:n, k in K if i != j))
    @constraint(model, [i in 2:n], sum(x[i,j,k] for j in 1:n, k in K if j != i) == 1)
    @constraint(model, [j in 2:n, k in K], sum(x[i,j,k] for i in 1:n if i != j) == sum(x[j,i,k] for i in 1:n if i != j))
    @constraint(model, [k in K], sum(x[1,j,k] for j in 2:n) <= 1)
    @constraint(model, [k in K], sum(x[i,1,k] for i in 2:n) == sum(x[1,j,k] for j in 2:n))
    @constraint(model, [k in K], sum(matriz_tempo[i,j] * x[i,j,k] for i in 1:n, j in 1:n if i != j) + coleta * sum(x[i,j,k] for i in 2:n, j in 1:n if i != j) <= jornada)
    @constraint(model, [i in 2:n, j in 2:n, k in K; i != j], u[i,k] - u[j,k] + p * x[i,j,k] <= p - 1)
    optimize!(model)
    return value.(x)
end

function highs_vrp_tw2(matriz_custo, matriz_tempo; jornada=480, coleta=60)
    n = size(matriz_custo, 1)
    p = n - 1
    K = 1:p
    M = jornada + maximum(matriz_tempo) + coleta
    model = Model(HiGHS.Optimizer)
    set_silent(model)
    @variable(model, x[1:n, 1:n, K], Bin)
    @variable(model, 0 <= tempo[2:n, K] <= jornada)
    @constraint(model, [i in 1:n, k in K], x[i,i,k] == 0)
    @objective(model, Min, sum(matriz_custo[i,j] * x[i,j,k] for i in 1:n, j in 1:n, k in K if i != j))
    @constraint(model, [i in 2:n], sum(x[i,j,k] for j in 1:n, k in K if j != i) == 1)
    @constraint(model, [j in 2:n, k in K], sum(x[i,j,k] for i in 1:n if i != j) == sum(x[j,i,k] for i in 1:n if i != j))
    @constraint(model, [k in K], sum(x[1,j,k] for j in 2:n) <= 1)
    @constraint(model, [k in K], sum(x[i,1,k] for i in 2:n) == sum(x[1,j,k] for j in 2:n))
    @constraint(model, [j in 2:n, k in K], tempo[j,k] >= matriz_tempo[1,j] + coleta - M * (1 - x[1,j,k]))
    @constraint(model, [i in 2:n, j in 2:n, k in K; i != j], tempo[j,k] >= tempo[i,k] + matriz_tempo[i,j] + coleta - M * (1 - x[i,j,k]))
    @constraint(model, [i in 2:n, k in K], tempo[i,k] >= coleta * sum(x[j,i,k] for j in 1:n if j != i))
    optimize!(model)
    return value.(x)
end

highs_vrp_tw2 (generic function with 1 method)

# TESTES

In [21]:
arquivo_csv = "resultados.csv"
criar_arquivo_csv("resultados.csv")
filepath = "Matrizes/matrizcusto10.json"

modelos = [
    ("HiGHS VRP CLASSICO", highs_tsp_classico, false),
    ("HiGHS VRP MTZ 1v", highs_tsp_mtz_1, false),
    ("HiGHS VRP MTZ 5v", highs_tsp_mtz_5, false),
    ("HiGHS VRP MTZ TW", highs_vrp_mtz_TW, true),
    ("HiGHS VRP TW2", highs_vrp_tw2, true)
]

for (nome_modelo, modelo, tw) in modelos
    println("Executando modelo: $nome_modelo")
    pipeline_highs(filepath, modelo, nome_modelo, tw)
end

Executando modelo: HiGHS VRP CLASSICO
Rotas HiGHS: Any[[1, 7, 4, 6, 8, 5, 10, 3, 9, 2, 1]]
Executando modelo: HiGHS VRP MTZ 1v
Rotas HiGHS: Any[[1, 7, 4, 6, 8, 5, 10, 3, 9, 2, 1]]
Executando modelo: HiGHS VRP MTZ 5v
Rotas HiGHS: Any[[1, 3, 1], [1, 4, 1], [1, 6, 8, 5, 10, 1], [1, 7, 1], [1, 9, 2, 1]]
Executando modelo: HiGHS VRP MTZ TW
Rotas HiGHS: Any[[1, 6, 5, 1], [1, 4, 8, 1], [1, 3, 9, 2, 1], [1, 7, 10, 1]]
Executando modelo: HiGHS VRP TW2
Rotas HiGHS: Any[[1, 7, 3, 9, 2, 1], [1, 4, 10, 1], [1, 5, 8, 6, 1]]
